# ALPHA构建
对于上述每一个组合，控制市值等因子，计算其ALPHA的显著性        

## 导入库

In [153]:
import warnings
from pathlib import Path
import polars as pl
import numpy as np
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS
import plotly.express as px

## 超参数

In [154]:
import os
import dotenv
dotenv.load_dotenv()

CONNECTION_URL = os.getenv("POSTGRES_URL")
ENGINE = "adbc"

TASK_ID_PREFIX = 'mech1'  # 任务id前缀
SAVE_BASE_DIR = f'/home/frank/files/programs/GraduationThesis/empirical/{TASK_ID_PREFIX}' # 保存基本路径
SAVE = True # 是否保存数据

RISK_FREE_RATE = 0.015 / 12 # 无风险利率  

## 读取数据(测试)
### 读取FF5数据  

FF5数据是一个月度时序数据，统计了每个月，FF5因子按照2*3方式构建组合的对冲组合的收益。使用该数据来计算alpha  

从json导入（测试）  

In [155]:
# 从 DB 读取 FF5（statics.stk_mkt_fivefacmonth）
ff5 = pl.read_database_uri(
    uri=CONNECTION_URL,
    query="""
        SELECT markettype_id AS "MarkettypeID", trading_month AS "TradingMonth",
               portfolios AS "Portfolios", risk_premium1 AS "RiskPremium1",
               smb1 AS "SMB1", hml1 AS "HML1", rmw1 AS "RMW1", cma1 AS "CMA1"
        FROM statics.stk_mkt_fivefacmonth
    """,
    engine=ENGINE,
)


### 读取分桶数据  
- 分桶数据用来和FF5数据合并，计算alpha     
- 分桶表现数据用于合并alpha表现  



In [156]:
combined_series = pl.read_parquet(SAVE_BASE_DIR + '/分桶组合收益.parquet')
performance = pl.read_parquet(SAVE_BASE_DIR + '/分桶表现.parquet')

In [157]:
performance

bucket_id,count,mean_return,sharp
str,u32,str,str
"""0""",241,""" -0.0008808 [-0.3703] (0…","""-0.03104"""
"""1""",241,""" 0.0001402 [0.1485] (0…","""0.01158"""
"""2""",241,""" -0.0001216 [-0.1958] (0…","""-0.01514"""
"""3""",241,""" 0.0001773 [0.3523] (0…","""0.02926"""
"""4""",241,""" 0.000148 [0.3617] (0…","""0.02878"""
…,…,…,…
"""6""",241,""" 0.0004683 [1.249] (0…","""0.09564"""
"""7""",241,""" 0.0007117 [1.998] (0.…","""0.1429"""
"""8""",241,""" 0.001345 [3.541] (0.0…","""0.259"""


In [158]:
combined_series.head()

date,weighted_sum_ret,bucket_id
date,f64,str
2004-12-01,-0.013505,"""0"""
2004-12-01,-0.007534,"""1"""
2004-12-01,-0.009161,"""2"""
2004-12-01,-0.007407,"""3"""
2004-12-01,-0.003221,"""4"""


In [159]:
performance

bucket_id,count,mean_return,sharp
str,u32,str,str
"""0""",241,""" -0.0008808 [-0.3703] (0…","""-0.03104"""
"""1""",241,""" 0.0001402 [0.1485] (0…","""0.01158"""
"""2""",241,""" -0.0001216 [-0.1958] (0…","""-0.01514"""
"""3""",241,""" 0.0001773 [0.3523] (0…","""0.02926"""
"""4""",241,""" 0.000148 [0.3617] (0…","""0.02878"""
…,…,…,…
"""6""",241,""" 0.0004683 [1.249] (0…","""0.09564"""
"""7""",241,""" 0.0007117 [1.998] (0.…","""0.1429"""
"""8""",241,""" 0.001345 [3.541] (0.0…","""0.259"""


## 处理数据：  

- 1.选用Portfolio == 1 的组合 (2*3构建)  
- 2.MarkettypeID == P9725：沪深A股和创业板和科创板 (不选京，否则没有早期年份)     
- 3.去除上述两列  
- 4.将TradingMonth列重命名为`date`    
- 5.因子重命名为`market_ret, smb, hml, rmw, cma`  

In [160]:
ff5 = ff5.filter((pl.col('Portfolios') == 1) & (pl.col('MarkettypeID') == 'P9714')).select(['TradingMonth', 'RiskPremium1', 'SMB1', 'HML1', 'RMW1', 'CMA1'])
ff5 = ff5.rename({'TradingMonth':'date', 'RiskPremium1':'market_ret', 'SMB1':'smb', 'HML1':'hml', 'RMW1':'rmw', 'CMA1':'cma'})

In [161]:
ff5.head()

date,market_ret,smb,hml,rmw,cma
date,f64,f64,f64,f64,f64
1997-04-01,0.08511,-0.047831,-0.028225,0.000637,-0.166661
1997-08-01,-0.001449,0.008266,0.047883,-0.082115,0.077224
1997-09-01,-0.116536,0.022238,-0.007937,-0.00569,-0.001345
1997-07-01,-0.066855,0.045986,0.012023,0.004122,0.014512
1997-06-01,-0.004782,0.015851,0.008128,0.044233,-0.054761


## 计算alpha  

### CAPM-ALPHA    
从FF5中获取market_ret列，形成时序数据`date-market_ret`  
将数据和combined_series合并，形成`date-code-bucket_id-ret-market_ret`表  

计算`(ret-risk_free_rate)~market_ret`回归的alpha（NW标准误）   

In [162]:
# 获取date-market_ret
market_ret = ff5.select('date','market_ret')

# 合并
joined_series = combined_series.join(market_ret, on='date', how='left')

In [163]:
def regress_one(s: pl.DataFrame) -> pl.DataFrame:
    g = s.to_pandas()
    bid = g['bucket_id'].iloc[0]  # 安全取分组 id，避免 Polars 内索引触发 panic
    bid = str(bid) 
    try:
        y = g['weighted_sum_ret'].to_numpy() - RISK_FREE_RATE
        x = g['market_ret'].to_numpy()
        x_with_constant = sm.add_constant(x)
        model = sm.OLS(y, x_with_constant)
        results = model.fit() if len(y) < 10 else model.fit(cov_type='HAC', cov_kwds={'maxlags': 12})
        return pl.DataFrame({
            'bucket_id': [bid],
            'camp-alpha': [float(results.params[0])],
            't': [float(results.tvalues[0])],
            'p': [float(results.pvalues[0])],
        })
    except Exception as e:
        import traceback
        traceback.print_exc()
        warnings.warn(f"计算{bid}时发生错误: {e}")
        return pl.DataFrame({
            'bucket_id': [bid],
            'camp-alpha': [0.0],
            't': [0.0],
            'p': [1.0],
        })

alpha_table = joined_series.group_by('bucket_id').map_groups(regress_one)

格式化输出：   
[ ]内为HAC-t，()内为p值    

In [164]:
# 格式化输出（4 位有效数字）
alpha_table = alpha_table.with_columns(
    pl.col('camp-alpha').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('camp-alpha'),
    pl.col('t').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('t'),
    pl.col('p').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('p'),
)

# 为t和p添加括号  
alpha_table = alpha_table.select(
    pl.col('bucket_id'),
    pl.col('camp-alpha'),
    (pl.lit('[') + pl.col('t') + pl.lit(']')).alias('t'),
    (pl.lit('(') + pl.col('p') + pl.lit(')')).alias('p'),
)

# 将alpha、t、p居中对齐
alpha_table = alpha_table.with_columns(
    pl.col('camp-alpha').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('camp-alpha'),
    pl.col('t').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('t'),
    pl.col('p').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('p'),
)

# 将alpha、t、p合并为一行
alpha_table = alpha_table.select(
    pl.col('bucket_id'),
    (pl.col('camp-alpha') + pl.lit('\n') + pl.col('t') + pl.lit('\n') + pl.col('p')).alias('camp-alpha'),
)

alpha_table = alpha_table.sort('bucket_id')

with pl.Config(tbl_width_chars=200, tbl_cols=20, fmt_str_lengths=200):
    display(alpha_table)

bucket_id,camp-alpha
str,str
"""0""",""" -0.002298 [-1.1] (0.2715) """
"""1""",""" -0.001225 [-1.541] (0.1232) """
"""2""",""" -0.001427 [-2.659] (0.007827) """
"""3""",""" -0.001131 [-2.623] (0.008705) """
"""4""",""" -0.001169 [-3.43] (0.0006033) """
…,…
"""6""",""" -0.0008239 [-2.501] (0.01238) """
"""7""",""" -0.0006031 [-2.013] (0.04416) """
"""8""",""" 2.526e-05 [0.08109] (0.9354) """


### FF3-ALPHA  
使用FAMA-FRENCH模型计算ALPHA  

与CAPM类似，使用`market_ret, smb, hml, rmw, cma`作为自变量，计算`weighted_sum_ret`的alpha    

In [165]:
# 获取ff3数据
ff3 = ff5.select('date', 'market_ret', 'smb', 'hml')

#连接
joined_series = combined_series.join(ff3, on='date', how='left')

# 按bucket_id分组回归
def regress_ff3(s: pl.DataFrame) -> pl.DataFrame:
    g = s.to_pandas()
    bid = g['bucket_id'].iloc[0]
    try:
        y = g['weighted_sum_ret'].to_numpy() - RISK_FREE_RATE
        x = g[['market_ret', 'smb', 'hml']].to_numpy()
        x_with_constant = sm.add_constant(x)
        results = sm.OLS(y, x_with_constant).fit(cov_type='HAC', cov_kwds={'maxlags': 12})
        return pl.DataFrame({
            'bucket_id': [bid],
            'ff3-alpha': [float(results.params[0])],
            't': [float(results.tvalues[0])],
            'p': [float(results.pvalues[0])],
        })
    except Exception as e:
        warnings.warn(f"计算{bid}时发生错误: {e}")
        return pl.DataFrame({
            'bucket_id': [bid],
            'ff3-alpha': [0.0],
            't': [0.0],
            'p': [1.0],
        })

ff3_alpha_table = joined_series.group_by('bucket_id').map_groups(regress_ff3)

格式化输出

In [166]:
# FF3-alpha 格式化输出（4 位有效数字、括号、居中对齐、合并一行）
ff3_alpha_table = ff3_alpha_table.with_columns(
    pl.col('ff3-alpha').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('ff3-alpha'),
    pl.col('t').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('t'),
    pl.col('p').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('p'),
)
ff3_alpha_table = ff3_alpha_table.select(
    pl.col('bucket_id'),
    pl.col('ff3-alpha'),
    (pl.lit('[') + pl.col('t') + pl.lit(']')).alias('t'),
    (pl.lit('(') + pl.col('p') + pl.lit(')')).alias('p'),
)
ff3_alpha_table = ff3_alpha_table.with_columns(
    pl.col('ff3-alpha').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('ff3-alpha'),
    pl.col('t').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('t'),
    pl.col('p').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('p'),
)
ff3_alpha_table = ff3_alpha_table.select(
    pl.col('bucket_id'),
    (pl.col('ff3-alpha') + pl.lit('\n') + pl.col('t') + pl.lit('\n') + pl.col('p')).alias('ff3-alpha'),
)

ff3_alpha_table = ff3_alpha_table.sort('bucket_id')

with pl.Config(tbl_width_chars=200, tbl_cols=20, fmt_str_lengths=200):
    display(ff3_alpha_table)


bucket_id,ff3-alpha
str,str
"""0""",""" -0.002011 [-0.9882] (0.3231) """
"""1""",""" -0.001012 [-1.284] (0.199) """
"""2""",""" -0.001329 [-2.516] (0.01188) """
"""3""",""" -0.001008 [-2.516] (0.01185) """
"""4""",""" -0.001093 [-3.3] (0.0009659) """
…,…
"""6""",""" -0.0007353 [-2.411] (0.01589) """
"""7""",""" -0.0005264 [-1.924] (0.05432) """
"""8""",""" 9.997e-05 [0.3287] (0.7424) """


### FF5-ALPHA 

In [167]:
# 获取 FF5 数据（五因子）
ff5_factors = ff5.select('date', 'market_ret', 'smb', 'hml', 'rmw', 'cma')

# 连接
joined_series = combined_series.join(ff5_factors, on='date', how='left')

# 按 bucket_id 分组回归
def regress_ff5(s: pl.DataFrame) -> pl.DataFrame:
    g = s.to_pandas()
    bid = g['bucket_id'].iloc[0]
    try:
        y = g['weighted_sum_ret'].to_numpy() - RISK_FREE_RATE
        x = g[['market_ret', 'smb', 'hml', 'rmw', 'cma']].to_numpy()
        x_with_constant = sm.add_constant(x)
        model = sm.OLS(y, x_with_constant)
        results = model.fit() if len(y) < 10 else model.fit(cov_type='HAC', cov_kwds={'maxlags': 12})
        return pl.DataFrame({
            'bucket_id': [bid],
            'ff5-alpha': [float(results.params[0])],
            't': [float(results.tvalues[0])],
            'p': [float(results.pvalues[0])],
        })
    except Exception as e:
        warnings.warn(f"计算{bid}时发生错误: {e}")
        return pl.DataFrame({
            'bucket_id': [bid],
            'ff5-alpha': [0.0],
            't': [0.0],
            'p': [1.0],
        })

ff5_alpha_table = joined_series.group_by('bucket_id').map_groups(regress_ff5)

格式化输出

In [168]:
# FF5-alpha 格式化输出（4 位有效数字、括号、居中对齐、合并一行）
ff5_alpha_table = ff5_alpha_table.with_columns(
    pl.col('ff5-alpha').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('ff5-alpha'),
    pl.col('t').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('t'),
    pl.col('p').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('p'),
)
ff5_alpha_table = ff5_alpha_table.select(
    pl.col('bucket_id'),
    pl.col('ff5-alpha'),
    (pl.lit('[') + pl.col('t') + pl.lit(']')).alias('t'),
    (pl.lit('(') + pl.col('p') + pl.lit(')')).alias('p'),
)
ff5_alpha_table = ff5_alpha_table.with_columns(
    pl.col('ff5-alpha').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('ff5-alpha'),
    pl.col('t').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('t'),
    pl.col('p').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('p'),
)
ff5_alpha_table = ff5_alpha_table.select(
    pl.col('bucket_id'),
    (pl.col('ff5-alpha') + pl.lit('\n') + pl.col('t') + pl.lit('\n') + pl.col('p')).alias('ff5-alpha'),
)

ff5_alpha_table = ff5_alpha_table.sort('bucket_id')

with pl.Config(tbl_width_chars=200, tbl_cols=20, fmt_str_lengths=200):
    display(ff5_alpha_table)

bucket_id,ff5-alpha
str,str
"""0""",""" -0.001593 [-0.8128] (0.4163) """
"""1""",""" -0.0008514 [-1.129] (0.2588) """
"""2""",""" -0.001124 [-2.27] (0.02321) """
"""3""",""" -0.0008335 [-2.146] (0.03188) """
"""4""",""" -0.000962 [-3.056] (0.002246) """
…,…
"""6""",""" -0.0005744 [-1.92] (0.05488) """
"""7""",""" -0.0003641 [-1.427] (0.1537) """
"""8""",""" 0.000263 [0.8844] (0.3765) """


## 合并所有表现

In [169]:
performance_all = performance.join(alpha_table, on='bucket_id', how='left')
performance_all = performance_all.join(ff3_alpha_table, on='bucket_id', how='left')
performance_all = performance_all.join(ff5_alpha_table, on='bucket_id', how='left')
performance_all = performance_all.sort('bucket_id')

In [170]:
with pl.Config(
    tbl_rows=50,           # 显示所有行（不截断高度）
    tbl_cols=50,           # 显示所有列
    tbl_width_chars=200,     # 不限制表格总宽度
    fmt_str_lengths=200,     # 不限制字符串长度
):
    display(performance_all)  # 显示整个完整的表


bucket_id,count,mean_return,sharp,camp-alpha,ff3-alpha,ff5-alpha
str,u32,str,str,str,str,str
"""0""",241,""" -0.0008808 [-0.3703] (0.7111) ""","""-0.03104""",""" -0.002298 [-1.1] (0.2715) """,""" -0.002011 [-0.9882] (0.3231) """,""" -0.001593 [-0.8128] (0.4163) """
"""1""",241,""" 0.0001402 [0.1485] (0.882) ""","""0.01158""",""" -0.001225 [-1.541] (0.1232) """,""" -0.001012 [-1.284] (0.199) """,""" -0.0008514 [-1.129] (0.2588) """
"""2""",241,""" -0.0001216 [-0.1958] (0.8448) ""","""-0.01514""",""" -0.001427 [-2.659] (0.007827) """,""" -0.001329 [-2.516] (0.01188) """,""" -0.001124 [-2.27] (0.02321) """
"""3""",241,""" 0.0001773 [0.3523] (0.7246) ""","""0.02926""",""" -0.001131 [-2.623] (0.008705) """,""" -0.001008 [-2.516] (0.01185) """,""" -0.0008335 [-2.146] (0.03188) """
"""4""",241,""" 0.000148 [0.3617] (0.7176) ""","""0.02878""",""" -0.001169 [-3.43] (0.0006033) """,""" -0.001093 [-3.3] (0.0009659) """,""" -0.000962 [-3.056] (0.002246) """
"""5""",241,""" 0.0002432 [0.5987] (0.5494) ""","""0.0477""",""" -0.001059 [-2.996] (0.002737) """,""" -0.0009791 [-3.041] (0.002355) """,""" -0.0008096 [-2.641] (0.008264) """
"""6""",241,""" 0.0004683 [1.249] (0.2118) ""","""0.09564""",""" -0.0008239 [-2.501] (0.01238) """,""" -0.0007353 [-2.411] (0.01589) """,""" -0.0005744 [-1.92] (0.05488) """
"""7""",241,""" 0.0007117 [1.998] (0.04576) ""","""0.1429""",""" -0.0006031 [-2.013] (0.04416) """,""" -0.0005264 [-1.924] (0.05432) """,""" -0.0003641 [-1.427] (0.1537) """
"""8""",241,""" 0.001345 [3.541] (0.0003987) ""","""0.259""",""" 2.526e-05 [0.08109] (0.9354) """,""" 9.997e-05 [0.3287] (0.7424) """,""" 0.000263 [0.8844] (0.3765) """


In [171]:
if SAVE:
    performance_all.write_parquet(SAVE_BASE_DIR + '/基准回归-分桶表现+alpha.parquet')